In [147]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch import nn

In [148]:
df = pd.read_csv("test.csv")
df.head()
X, y = df["X"], df["y"]

X = torch.from_numpy(X.to_numpy()).type(dtype=torch.float32)
y = torch.from_numpy(y.to_numpy()).type(dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [149]:
class Regularization_Term(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(1))
        self.bias = nn.Parameter(torch.rand(1))
    def forward(self,X):
        return self.weight*X+self.bias

In [150]:
def Acc_Func(y_true, y_pred):
    return (torch.eq(y_true, y_pred).sum().item() / len(y_true)) * 100

In [151]:
model30 = Regularization_Term()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model30.parameters(), lr=0.001, momentum=0.9,weight_decay=0.1)
epochs = 200    

lambda_L2 = 0.01
lambda_L1 = 0.01
for epoch in range(epochs):
    model30.train()
    y_preds = model30(X_train)
    loss = loss_fn(y_preds, y_train) + lambda_L1 * torch.sum(torch.abs(model30.weight))+ lambda_L2*torch.sum(model30.weight**2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model30.eval()
    with torch.inference_mode():
        test_preds = model30(X_test)
        test_loss = loss_fn(test_preds, y_test)

        # Round predictions to nearest integer
        y_pred_round = torch.round(test_preds)

        acc = Acc_Func(y_test, y_pred_round)
        if epoch % 20 == 0:
            print(
                f"Epoch: {epoch}, Loss: {loss:.6f}, Test Loss: {test_loss:.6f}, Accuracy: {acc:.2f}%"
            )

Epoch: 0, Loss: 147.299744, Test Loss: 41.553177, Accuracy: 0.00%
Epoch: 20, Loss: 15.556208, Test Loss: 6.480436, Accuracy: 0.00%
Epoch: 40, Loss: 0.999631, Test Loss: 0.809651, Accuracy: 33.33%
Epoch: 60, Loss: 0.076175, Test Loss: 0.021696, Accuracy: 100.00%
Epoch: 80, Loss: 0.068046, Test Loss: 0.011554, Accuracy: 100.00%
Epoch: 100, Loss: 0.069348, Test Loss: 0.010341, Accuracy: 100.00%
Epoch: 120, Loss: 0.066365, Test Loss: 0.007343, Accuracy: 100.00%
Epoch: 140, Loss: 0.066482, Test Loss: 0.008907, Accuracy: 100.00%
Epoch: 160, Loss: 0.066123, Test Loss: 0.007993, Accuracy: 100.00%
Epoch: 180, Loss: 0.066083, Test Loss: 0.008122, Accuracy: 100.00%


In [152]:
print(model30.weight.item())
print(model30.bias.item())

2.012962818145752
0.8440102934837341


In [153]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test.numpy(),test_preds.numpy())
print(f"R2 Score: {r2:.4f}")

R2 Score: 0.9997


In [154]:
for name, param in model30.named_parameters():
    if param.grad is not None:
        print(name, param.grad)

weight tensor([-0.2042])
bias tensor([-0.0897])
